In [12]:
import fitz  # PyMuPDF
import os

pdf_path = "./Meg_proj.pdf"
out_dir = "images"
os.makedirs(out_dir, exist_ok=True)

doc = fitz.open(pdf_path)

print("Number of pages in PDF:", len(doc))
img_count = 0

for page_index in range(len(doc)):
    page = doc[page_index]
    images = page.get_images(full=True)

    for img in images:
        xref = img[0]
        base = doc.extract_image(xref)
        image_bytes = base["image"]
        ext = base["ext"]

        fname = f"{out_dir}/img_{img_count:04d}.{ext}"
        with open(fname, "wb") as f:
            f.write(image_bytes)

        img_count += 1

print("Extracted", img_count, "images")

Number of pages in PDF: 208
Extracted 208 images


In [13]:
import os, shutil, glob
import numpy as np
from PIL import Image

src = "../png"
out = "../png_sub"
os.makedirs(out, exist_ok=True)

paths = sorted(glob.glob(os.path.join(src, "*.png")))
print("Images:", len(paths))

def load_gray_small(p, scale=0.25):
    im = Image.open(p).convert("L")
    w, h = im.size
    im = im.resize((max(32, int(w*scale)), max(32, int(h*scale))), Image.Resampling.BILINEAR)
    return np.asarray(im, dtype=np.float32)

def blur_score(p):
    # Laplacian variance without cv2
    x = load_gray_small(p, 0.25)
    # 2D Laplacian kernel
    k = np.array([[0, 1, 0],
                  [1,-4, 1],
                  [0, 1, 0]], dtype=np.float32)
    # convolution (valid)
    y = (
        k[0,0]*x[:-2,:-2] + k[0,1]*x[:-2,1:-1] + k[0,2]*x[:-2,2:] +
        k[1,0]*x[1:-1,:-2] + k[1,1]*x[1:-1,1:-1] + k[1,2]*x[1:-1,2:] +
        k[2,0]*x[2:,:-2] + k[2,1]*x[2:,1:-1] + k[2,2]*x[2:,2:]
    )
    return float(np.var(y))

def dhash(p, hash_size=16):
    # difference hash (very fast, robust enough for grouping)
    im = Image.open(p).convert("L").resize((hash_size+1, hash_size), Image.Resampling.BILINEAR)
    arr = np.asarray(im, dtype=np.float32)
    diff = arr[:,1:] > arr[:,:-1]
    # pack to bytes
    return diff.flatten()

def hamming(a, b):
    return int(np.count_nonzero(a != b))

# 1) keep sharpest 70%
scores = [(p, blur_score(p)) for p in paths]
scores.sort(key=lambda x: x[1], reverse=True)
keep = scores[: max(20, int(0.7*len(scores)))]
keep_paths = [p for p,_ in keep]
print("Kept (sharp):", len(keep_paths))

# 2) build similarity graph via dhash
hashes = {p: dhash(p) for p in keep_paths}

# Compare only within sliding window to keep it fast.
# If your filenames are mixed, this window may miss links;
# set window=None to compare all pairs (slower but safer for 200 imgs).
window = None  # set to e.g. 40 if your list is roughly grouped

edges = {p: [] for p in keep_paths}
thr = 55  # smaller = stricter (more similar). 55/256 is moderate.

if window is None:
    for i,p in enumerate(keep_paths):
        for j in range(i+1, len(keep_paths)):
            q = keep_paths[j]
            if hamming(hashes[p], hashes[q]) <= thr:
                edges[p].append(q); edges[q].append(p)
else:
    for i,p in enumerate(keep_paths):
        for j in range(max(0, i-window), min(len(keep_paths), i+window+1)):
            if i == j: continue
            q = keep_paths[j]
            if hamming(hashes[p], hashes[q]) <= thr:
                edges[p].append(q); edges[q].append(p)

# 3) largest connected component (BFS)
seen = set()
best = []
for p in keep_paths:
    if p in seen: continue
    comp = []
    stack = [p]
    seen.add(p)
    while stack:
        u = stack.pop()
        comp.append(u)
        for v in edges[u]:
            if v not in seen:
                seen.add(v); stack.append(v)
    if len(comp) > len(best):
        best = comp

print("Largest component:", len(best))
for p in best:
    shutil.copy2(p, os.path.join(out, os.path.basename(p)))
print("Wrote subset to:", out)

Images: 0
Kept (sharp): 0
Largest component: 0
Wrote subset to: ../png_sub


colmap feature_extractor --database_path db.db --image_path . --FeatureExtraction.use_gpu 0 --ImageReader.single_camera 1

colmap exhaustive_matcher --database_path db.db --FeatureMatching.use_gpu 0